# Обучение LLM "Pretrain + SFT"

## Установка зависимостей

In [1]:
# !pip install transformers==4.55.2 torch==2.8.0 numpy==1.26.1 datasets==4.0.0 accelerate==1.10.0 bitsandbytes==0.47.0 trl==0.22.2 jinja2==3.1.4 tokenizers sentencepiece tqdm -q

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [2]:
import os
import re
import glob
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset, load_dataset
from transformers import (
    PreTrainedTokenizerFast,
    LlamaConfig,
    LlamaForCausalLM,
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    TrainerCallback
)
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, processors, decoders
from trl import SFTTrainer, SFTConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [3]:
!nvidia-smi

Sat Jan 24 22:37:34 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.247.01             Driver Version: 535.247.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  | 00000000:8B:00.0 Off |                    0 |
| N/A   31C    P0              74W / 500W |      3MiB / 81920MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

## 1.  PRETRAIN

### 1.1 Скачивание данных RussianNovels

In [ ]:
# !git clone https://github.com/JoannaBy/RussianNovels.git

In [5]:
# Загрузка всех текстов
corpus_path = "RussianNovels/corpus"
texts = []

for filepath in glob.glob(os.path.join(corpus_path, "*.txt")):
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
        texts.append(text)

print(f"Загружено {len(texts)} произведений")
print(f"Общий объём: {sum(len(t) for t in texts):,} символов")

Загружено 108 произведений
Общий объём: 45,348,575 символов


### 1.2 Препроцессинг данных

In [6]:
def is_cyrillic_sentence(sentence):
    """Проверка что предложение содержит только кириллицу и базовую пунктуацию"""
    # Разрешаем кириллицу, пробелы, знаки препинания, цифры
    allowed = re.compile(r'^[а-яА-ЯёЁ\s\d.,!?;:\-\—\–\'\"«»()\[\]…]+$')
    return bool(allowed.match(sentence)) and len(sentence.strip()) > 0

def clean_text(text):
    """Очистка текста"""
    # Замена множественной пунктуации
    text = re.sub(r'[.]{2,}', '…', text)
    text = re.sub(r'[!]{2,}', '!', text)
    text = re.sub(r'[?]{2,}', '?', text)
    text = re.sub(r'[-]{2,}', '—', text)
    
    # Убираем лишние пробелы
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n+', '\n\n', text)
    
    return text.strip()

def split_into_sentences(text):
    """Разбивка на предложения"""
    sentences = re.split(r'(?<=[.!?…])\s+', text)
    return [s.strip() for s in sentences if s.strip()]

# Препроцессинг
all_sentences = []
for text in tqdm(texts, desc="Preprocessing"):
    text = clean_text(text)
    sentences = split_into_sentences(text)
    for sent in sentences:
        if is_cyrillic_sentence(sent) and len(sent) > 10:
            all_sentences.append(sent)

# Удаление дубликатов
all_sentences = list(set(all_sentences))
print(f"После препроцессинга: {len(all_sentences)} уникальных предложений")

Preprocessing: 100%|██████████| 108/108 [00:05<00:00, 19.86it/s]


После препроцессинга: 465978 уникальных предложений


In [7]:
# Объединяем предложения в чанки для обучения
MAX_CHUNK_CHARS = 1500  # примерно 512 токенов

chunks = []
current_chunk = []
current_len = 0

for sent in all_sentences:
    if current_len + len(sent) > MAX_CHUNK_CHARS and current_chunk:
        chunks.append(' '.join(current_chunk))
        current_chunk = [sent]
        current_len = len(sent)
    else:
        current_chunk.append(sent)
        current_len += len(sent)

if current_chunk:
    chunks.append(' '.join(current_chunk))

print(f"Создано {len(chunks)} чанков")
print(f"Средняя длина чанка: {np.mean([len(c) for c in chunks]):.0f} символов")

Создано 26638 чанков
Средняя длина чанка: 1441 символов


### 1.3 Создание и обучение токенизатора (BPE)

In [8]:
# Сохраняем тексты для обучения токенизатора
with open("train_texts.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(chunks))

# Создание BPE токенизатора
tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

# Специальные токены
special_tokens = ["<pad>", "<bos>", "<eos>", "<unk>"]

trainer = trainers.BpeTrainer(
    vocab_size=3000,
    special_tokens=special_tokens,
    min_frequency=2
)

tokenizer.train(["train_texts.txt"], trainer)

# Добавляем постпроцессор для BOS/EOS
tokenizer.post_processor = processors.TemplateProcessing(
    single="<bos> $A <eos>",
    special_tokens=[
        ("<bos>", tokenizer.token_to_id("<bos>")),
        ("<eos>", tokenizer.token_to_id("<eos>")),
    ],
)

# Сохраняем
tokenizer.save("russian_novels_tokenizer.json")
print(f"Словарь: {tokenizer.get_vocab_size()} токенов")




Словарь: 3000 токенов


In [9]:
# Оборачиваем в HuggingFace токенизатор
from transformers import PreTrainedTokenizerFast

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="russian_novels_tokenizer.json",
    bos_token="<bos>",
    eos_token="<eos>",
    unk_token="<unk>",
    pad_token="<pad>",
)

# Проверка
test_text = "Все счастливые семьи похожи друг на друга."
encoded = hf_tokenizer(test_text)
print(f"Пример токенизации: {test_text}")
print(f"Токены: {hf_tokenizer.convert_ids_to_tokens(encoded['input_ids'])}")

Пример токенизации: Все счастливые семьи похожи друг на друга.
Токены: ['<bos>', 'ÐĴ', 'ÑģÐµ', 'ĠÑģÑĩÐ°ÑģÑĤÐ»Ð¸Ð²', 'ÑĭÐµ', 'ĠÑģÐµÐ¼', 'ÑĮÐ¸', 'ĠÐ¿Ð¾ÑħÐ¾Ð¶', 'Ð¸', 'ĠÐ´ÑĢÑĥÐ³', 'ĠÐ½Ð°', 'ĠÐ´ÑĢÑĥÐ³Ð°', '.', '<eos>']


### 1.4 Подготовка датасета для претрейна

In [10]:
MAX_LENGTH = 512

def tokenize_function(examples):
    return hf_tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors=None
    )

# Создаём Dataset
dataset = Dataset.from_dict({"text": chunks})
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# Добавляем labels для CLM
def add_labels(examples):
    examples["labels"] = examples["input_ids"].copy()
    return examples

tokenized_dataset = tokenized_dataset.map(add_labels, batched=True)
tokenized_dataset.set_format("torch")

print(f"Dataset size: {len(tokenized_dataset)}")
print(f"Sample keys: {tokenized_dataset[0].keys()}")

Map: 100%|██████████| 26638/26638 [00:06<00:00, 4231.01 examples/s]

Dataset size: 26638
Sample keys: dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])


### 1.5 Инициализация модели (~150M параметров)

In [11]:
config = LlamaConfig(
    vocab_size=hf_tokenizer.vocab_size,
    hidden_size=1024,
    intermediate_size=1536,
    num_hidden_layers=16,
    num_attention_heads=16,
    num_key_value_heads=8,
    max_position_embeddings=MAX_LENGTH,
    pad_token_id=hf_tokenizer.pad_token_id,
    bos_token_id=hf_tokenizer.bos_token_id,
    eos_token_id=hf_tokenizer.eos_token_id,
)

model = LlamaForCausalLM(config)
num_params = sum(p.numel() for p in model.parameters())
print(f"Модель создана: {num_params / 1e6:.1f}M параметров")

Модель создана: 132.0M параметров


### 1.6 Коллбэк для валидации на тестовых промптах

In [12]:
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
]

class GenerationCallback(TrainerCallback):
    def __init__(self, tokenizer, prompts, device, every_n_steps=500):
        self.tokenizer = tokenizer
        self.prompts = prompts
        self.device = device
        self.every_n_steps = every_n_steps
    
    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step % self.every_n_steps == 0 and state.global_step > 0:
            model.eval()
            print(f"\n=== Generation at step {state.global_step} ===")
            for prompt in self.prompts[:3]:  # Показываем 3 примера
                inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
                # Убираем token_type_ids если есть
                inputs = {k: v for k, v in inputs.items() if k != "token_type_ids"}
                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=50,
                        do_sample=True,
                        temperature=0.7,
                        top_p=0.9,
                        pad_token_id=self.tokenizer.pad_token_id
                    )
                generated = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
                print(f"Prompt: {prompt}")
                print(f"Generated: {generated}\n")
            model.train()

### 1.7 Обучение Pretrain модели

In [13]:
training_args = TrainingArguments(
    output_dir="./pretrain_output",
    overwrite_output_dir=True,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=8,  # effective batch_size = 64
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    dataloader_num_workers=2,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=hf_tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    callbacks=[GenerationCallback(hf_tokenizer, test_prompts, device, every_n_steps=500)]
)

print("Начинаем обучение Pretrain модели...")
trainer.train()

Начинаем обучение Pretrain модели...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Step,Training Loss
50,7.001000
100,5.969000
150,5.186000
200,4.649300
250,4.294100
300,4.068000
350,3.925100
400,3.828900
450,3.709700
500,3.631900


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



=== Generation at step 500 ===
Prompt: Все мысли, которые имеют огромные последствия
Generated: Все мысли, которые имеют огромные последствия, — все кончено, — и он уже успел подойти к нему. — Я не забуду, — сказал он. Безбедов не снимал с себя глаз, а в глазах его, отступившего

Prompt: Сила войска зависит от его духа
Generated: Сила войска зависит от его духаВладимир Иванович Самгин понимал, что она не понимает, что с ним сделало
 свое, и что у него все было, что он делал. — А я, — вдруг повторил он, — у меня нет?

Prompt: Мысль о том, что он принес страдания
Generated: Мысль о том, что он принес страдания, в тот же день, когда его разорвали, он был приготовлен в Петербурге, но в первый раз в жизни, что он входил в комнату, в котором, несмотря на то, что ему удалось вести, и



...

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



=== Generation at step 1500 ===
Prompt: Все мысли, которые имеют огромные последствия
Generated: Все мысли, которые имеют огромные последствияий, и которые, хотя он и не был способен оправдывать, он считал, что все это только вредное, а не физическое, как бы фальшивое, а не даром что

Prompt: Сила войска зависит от его духа
Generated: Сила войска зависит от его духае, от
тебя от этого отступления в поле сражения. Безобразие, которое она уже вычеркнула, было так, как я увидал ее, — словно она была обманута.

Prompt: Мысль о том, что он принес страдания
Generated: Мысль о том, что он принес страдания, заставляла его верить, что он намерен жениться на ней, и что он не может быть никак не мог отказаться от его присутствия. — Да, — сказала я, — это хорошо, что у вас такое прекрас



huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)



=== Generation at step 2000 ===
Prompt: Все мысли, которые имеют огромные последствия
Generated: Все мысли, которые имеют огромные последствия, и сомнений, и сомнений, а не с тем, чтобы подвергать себя опасности. — Да, и вы были в полном уме, — с важностью ответил князь Андрей. Словом, его

Prompt: Сила войска зависит от его духа
Generated: Сила войска зависит от его духаательного движения. — А ты бы помог ей, Ардальон Борисыч, — заговорил он, — так, как все это было… - Идите, - сказал он, подходя к столу. — спросила она

Prompt: Мысль о том, что он принес страдания
Generated: Мысль о том, что он принес страдания, все, что он видел, все его мысли о том, что он не только не верил, но все-таки имел удовольствие думать, что он был очень рад. "Она, кажется, что-то не так?" — подумал



TrainOutput(global_step=2085, training_loss=3.434089872877089, metrics={'train_runtime': 1230.9188, 'train_samples_per_second': 108.204, 'train_steps_per_second': 1.694, 'total_flos': 5.275496733474816e+16, 'train_loss': 3.434089872877089, 'epoch': 5.0})

In [14]:
# Сохранение модели
trainer.save_model("./pretrain_final")
hf_tokenizer.save_pretrained("./pretrain_final")

('./pretrain_final/tokenizer_config.json',
 './pretrain_final/special_tokens_map.json',
 './pretrain_final/tokenizer.json')

### 1.8 Генерация на test_prompts (финальный результат)

In [15]:
model.eval()
model.to(device)

print("=" * 60)
print("РЕЗУЛЬТАТЫ PRETRAIN: Генерация на тестовых промптах")
print("=" * 60)

for i, prompt in enumerate(test_prompts, 1):
    inputs = hf_tokenizer(prompt, return_tensors="pt").to(device)
    inputs = {k: v for k, v in inputs.items() if k != "token_type_ids"}
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=hf_tokenizer.pad_token_id
        )
    
    generated = hf_tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\nPrompt {i}: {prompt}")
    print(f"Generated: {generated}")
    print("-" * 40)

РЕЗУЛЬТАТЫ PRETRAIN: Генерация на тестовых промптах

Prompt 1: Все мысли, которые имеют огромные последствия
Generated: Все мысли, которые имеют огромные последствия тому, какою он мог бы сделать; но в то же время ему хотелось сказать что-нибудь или другое о том, чтобы не мешало его право на все эти вопросы. – Помилуй, князь! Где ты был
бесчувствен? "И это я тоже думал, — думала она себе, — и вот тебе и стыдно. Он, вер
----------------------------------------

Prompt 2: Сила войска зависит от его духа
Generated: Сила войска зависит от его духае. — сказал он, хмуря брови.— Милый мой! Угрюмая, грязноватая ночь медленно проходила мимо нее и не шевелилась на ее лице; потом она вспомнила о своей жизни, о своем счастье… - Погодите, - продолжал Логин, - я сейчас, как вы изволили
 идти,-
----------------------------------------

Prompt 3: Мысль о том, что он принес страдания
Generated: Мысль о том, что он принес страдания или, вернее, только потому, что она была не такой, как эта книга его. Те

# 2. POST-TRAIN SFT


### 2.1 Загрузка инструктивного датасета alpaca-cleaned-ru

In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
alpaca_dataset = load_dataset("d0rj/alpaca-cleaned-ru", split="train")
print(f"Загружено {len(alpaca_dataset)} примеров")
print(f"Поля: {alpaca_dataset.features}")
print(f"\nПример:\n{alpaca_dataset[0]}")

Загружено 51760 примеров
Поля: {'input': Value('string'), 'instruction': Value('string'), 'output': Value('string')}

Пример:
{'input': '', 'instruction': 'Дайте три совета, как оставаться здоровым.', 'output': '1. Соблюдайте сбалансированную и питательную диету. Убедитесь, что в ваш рацион входят разнообразные фрукты и овощи, нежирный белок, цельнозерновые продукты и полезные жиры. Это помогает обеспечить ваш организм необходимыми питательными веществами для оптимального функционирования и может помочь предотвратить хронические заболевания.\n\n2. Занимайтесь регулярной физической активностью. Упражнения имеют решающее значение для поддержания крепких костей, мышц и здоровья сердечно-сосудистой системы. Старайтесь уделять не менее 150 минут умеренным аэробным упражнениям или 75 минут интенсивным упражнениям каждую неделю.\n\n3. Высыпайтесь. Достаточное количество качественного сна имеет решающее значение для физического и психического благополучия. Он помогает регулировать настроение, 

### 2.2 Подготовка данных

In [3]:
def format_as_text(example):
    text = ""
    if example.get("input") and example["input"].strip():
        text += f"Контекст: {example['input']}\n"
    text += f"Вопрос: {example['instruction']}\nОтвет: {example['output']}"
    return {"text": text}

text_dataset = alpaca_dataset.map(format_as_text, remove_columns=alpaca_dataset.column_names)
train_dataset = text_dataset.shuffle(seed=42).select(range(10000))
print(f"Пример:\n{train_dataset[0]['text'][:300]}")

Пример:
Контекст: Она рано ушла с вечеринки
Вопрос: Измените следующее предложение, чтобы сделать предложение более интересным.
Ответ: Рано она ушла с вечеринки.


### 2.3 Загрузка базовой модели Qwen2.5-0.5B

In [4]:
model_name = "Qwen/Qwen2.5-0.5B"

sft_tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
sft_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

sft_tokenizer.pad_token = sft_tokenizer.eos_token
sft_model.config.pad_token_id = sft_model.config.eos_token_id
print(f"Модель загружена: {model_name}")

Модель загружена: Qwen/Qwen2.5-0.5B


### 2.4 Генерация ДО SFT обучения

In [5]:
questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
]

def generate_response(model, tokenizer, prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=" * 60)
print("ГЕНЕРАЦИЯ ДО SFT (Base Model)")
print("=" * 60)

for i, q in enumerate(questions_rus, 1):
    response = generate_response(sft_model, sft_tokenizer, q)
    print(f"\nModel Input {i}:\n{q}")
    print(f"Model Output {i}:\n{response[len(q):]}")
    print("-" * 40)

ГЕНЕРАЦИЯ ДО SFT (Base Model)

Model Input 1:
сколько планет в нашей солнечной системе?
Model Output 1:
 - Семинар
Главная / Семинар / Семинар «Семинар» / Семинар «Семинар» / 1000-летний «Планетный океан» / 1000-летний «Планетный океан» / Солнечная система / Солнечная система / Солнечная система / Солнечная система / Солнечная система
----------------------------------------

Model Input 2:
расскажи стих
Model Output 2:
и о любви в стихах
Приветствую вас, мои читатели! Сегодня я расскажу вам стихи о любви, которые могут быть использованы в различных ситуациях. Стихи о любви могут быть использованы для написания песен, для развлечения, для поздравлений, для подарков, и для любого другого использования. Я расскажу вам стихи о люб
----------------------------------------

Model Input 3:
когда собирать крыжовник?
Model Output 3:
 - Постельные материалы | Новости
Крыжовник - это то, что называют ароматизированный поливинилхлорид, который подходит для создания розового или бархатного полотна

### 2.5 Токенизация с правильными labels

In [6]:
def tokenize_function(examples):
    tokens = sft_tokenizer(
        examples["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    # Важно: padding токены в labels = -100 (игнорируются в loss)
    labels = []
    for input_ids in tokens["input_ids"]:
        label = [token if token != sft_tokenizer.pad_token_id else -100 for token in input_ids]
        labels.append(label)
    tokens["labels"] = labels
    return tokens

tokenized_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_dataset.set_format("torch")
print("Датасет токенизирован")

Датасет токенизирован


### 2.6 SFT обучение с Trainer

С TRL SFTTrainer не получилось почему-то, был какой-то конфликт в версии библиотеки jinja.

In [7]:
training_args = TrainingArguments(
    output_dir="./sft_output",
    max_steps=300,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    save_steps=150,
    save_total_limit=2,
    bf16=True,
    report_to="none",
)

trainer = Trainer(
    model=sft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

print("Начинаем SFT обучение...")
trainer.train()

Начинаем SFT обучение...


Step,Training Loss
50,1.619800
100,1.550700
150,1.480100
200,1.518600
250,1.496800
300,1.468000


TrainOutput(global_step=300, training_loss=1.5223374938964844, metrics={'train_runtime': 123.8278, 'train_samples_per_second': 38.764, 'train_steps_per_second': 2.423, 'total_flos': 5277422400307200.0, 'train_loss': 1.5223374938964844, 'epoch': 0.48})

In [8]:
trainer.save_model("./sft_final")
sft_tokenizer.save_pretrained("./sft_final")
print("Модель сохранена!")

Модель сохранена!


### 2.7 Генерация ПОСЛЕ SFT обучения

In [9]:
questions_rus = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
]

print("=" * 60)
print("ГЕНЕРАЦИЯ ПОСЛЕ SFT (Fine-tuned Model)")
print("=" * 60)

sft_model.eval()

for i, q in enumerate(questions_rus, 1):
    prompt = f"Вопрос: {q}\nОтвет:"
    inputs = sft_tokenizer(prompt, return_tensors="pt").to(sft_model.device)
    
    with torch.no_grad():
        outputs = sft_model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            eos_token_id=sft_tokenizer.eos_token_id,
            pad_token_id=sft_tokenizer.pad_token_id,
        )
    
    response = sft_tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"\nModel Input {i}:\n{q}")
    print(f"Model Output {i}:\n{response[len(prompt):]}")
    print("-" * 40)

ГЕНЕРАЦИЯ ПОСЛЕ SFT (Fine-tuned Model)

Model Input 1:
сколько планет в нашей солнечной системе?
Model Output 1:
 На ours solar system существует 8 планет. Это планеты, которые orbitируют вокруг Солнца и обладают различными типами формирования, состояния и внешними особенностями. Одна из наиболее известных планет — Марс, которая находится недалеко от земли и имеет богатый континентом. Вторая планета, которой следует упомянуть, это Луция, а последняя
----------------------------------------

Model Input 2:
расскажи стих
Model Output 2:
 Вот стихотворение по-русски:

«Мы не можем борьбы,
Они мгновенно уходят в небо.
И мы знали бы только,
Что есть, когда ты будешь с нами».

Спасибо за внимание!
----------------------------------------

Model Input 3:
когда собирать крыжовник?
Model Output 3:
 Собирая крохотные крыжовники, обычно это занимает не более 30 минут. Взяться за их изготовление можно в течение нескольких часов до сбора. Это зависит от размера и количества крыжовников, что важно з